In [1]:
import os
import json
from tqdm import tqdm
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# ============================================================
# 1. CONFIGURATION
# ============================================================
INPUT_FILE = "experiment_dataset.jsonl"
OUTPUT_FILE = "selfcheck_dataset.jsonl"
OLLAMA_MODEL = "qwen2.5:7b"
N_SAMPLES = 3  # Number of high-temperature answers to check against

print("Initializing local Ollama models...")
# Strict judge model (greedy decoding for consistency)
judge_llm = ChatOllama(model=OLLAMA_MODEL, temperature=0.0)
# Creative/stochastic model for alternative sampling
sampler_llm = ChatOllama(model=OLLAMA_MODEL, temperature=1.0)

# Prompts for generation and evaluation
sample_prompt = ChatPromptTemplate.from_template("Question: {question}\nAnswer:")
judge_prompt = ChatPromptTemplate.from_template("""
You are analyzing a response for factual consistency against a set of alternative answers to detect hallucinations.

<target_answer>
{rag_answer}
</target_answer>

Here are {n_samples} alternative answers generated by the same model for the question: "{question}"
<alternative_samples>
{combined_samples}
</alternative_samples>

Does the <target_answer> contain facts, claims, or specific details that are completely unmentioned or contradicted by the majority of the alternative samples? 
If the target answer contains unverified or conflicting information compared to the samples, classify it as a hallucination.

Respond ONLY with a single digit:
1 (if a hallucination/inconsistency is detected)
0 (if the answer is consistent and verified by the samples)
""")

# ============================================================
# 2. CHECKPOINT ENGINE (Load Progress)
# ============================================================
processed_ids = set()

if os.path.exists(OUTPUT_FILE):
    print(f"🔄 Found existing progress file '{OUTPUT_FILE}'. Reading processed items...")
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                existing_data = json.loads(line)
                if "id" in existing_data:
                    processed_ids.add(str(existing_data["id"]))
            except json.JSONDecodeError:
                print(f"⚠️ Warning: Line {line_num} in '{OUTPUT_FILE}' is corrupted JSON. Skipping line.")
                
    print(f"⏮️ Found {len(processed_ids)} items already verified. They will be skipped automatically.\n")

# ============================================================
# 3. STREAM & EVALUATE DATASET
# ============================================================
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"❌ Input file '{INPUT_FILE}' not found! Please check the file path.")

# Pre-calculate total rows for an accurate progress bar
total_lines = 0
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip(): 
            total_lines += 1

print(f"🚀 Running SelfCheckGPT Consistency Detector...")

# Open output in append mode ('a') so it never overwrites existing records
with open(INPUT_FILE, 'r', encoding='utf-8') as infile, \
     open(OUTPUT_FILE, 'a', encoding='utf-8') as outfile:
    
    progress_bar = tqdm(infile, total=total_lines, desc="🕵️ SelfCheck GPT", unit="item")
    
    for line_num, line in enumerate(progress_bar, 1):
        line = line.strip()
        if not line:
            continue
            
        try:
            data = json.loads(line)
            item_id = str(data.get("id"))
            
            # ------------------------------------------------
            # RESUME TRIGGER: Skip if completed previously
            # ------------------------------------------------
            if item_id in processed_ids:
                continue
                
            question = data.get("question", "")
            rag_answer = data.get("rag_answer", "")
            
            # Skip rows missing a generated RAG answer
            if not rag_answer:
                continue
                
            progress_bar.set_postfix_str(f"Sampling: {item_id[:6]}...")
            
            # Step A: Generate Stochastic Samples
            samples = []
            for i in range(N_SAMPLES):
                response = (sample_prompt | sampler_llm).invoke({"question": question})
                samples.append(f"Sample {i+1}: {response.content.strip()}")
                
            combined_samples = "\n\n".join(samples)
            
            # Step B: LLM Judge consistency check
            progress_bar.set_postfix_str(f"Judging: {item_id[:6]}...")
            judge_response = (judge_prompt | judge_llm).invoke({
                "rag_answer": rag_answer,
                "question": question,
                "n_samples": N_SAMPLES,
                "combined_samples": combined_samples
            })
            
            # Extract binary token result
            raw_score = judge_response.content.strip()
            hallucination_score = 1 if "1" in raw_score else 0
            
            # Step C: Update dictionary and write instantly to disk
            data["selfcheck_samples"] = samples
            data["selfcheck_hallucination"] = hallucination_score
            
            # Write row to output file immediately
            outfile.write(json.dumps(data, ensure_ascii=False) + "\n")
            outfile.flush()  # Forces immediate hard drive commit

        except json.JSONDecodeError:
            print(f"\n⚠️ Malformed JSON at line {line_num} of input file. Skipping...")
        except Exception as e:
            print(f"\n❌ Error processing item {data.get('id', 'Unknown')}: {e}. Continuing to next item...")

print(f"\n🎉 Process complete! All persistent evaluations saved in '{OUTPUT_FILE}'.")

/Users/sterinsaji/miniconda3/envs/rag_project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing local Ollama models...
🚀 Running SelfCheckGPT Consistency Detector...


🕵️ SelfCheck GPT: 100%|██████████| 519/519 [7:54:53<00:00, 54.90s/item, Judging: c534ea...]   


🎉 Process complete! All persistent evaluations saved in 'selfcheck_dataset.jsonl'.
